# Databricks Notebook 2: Performance Optimization & Advanced Lakehouse Techniques
**Target Audience:** Azure Data Engineers & Senior Technical Interview Preparation  
**Objective:** Master hands-on PySpark optimization, storage format benchmarks, Delta Lake internals, and advanced data engineering patterns.

---

### Complete Curriculum (One Comprehensive Task Per Cell):
1. **Task 0: Interactive Setup & Environment Configuration**
2. **Task 1: Partitioning (`partitionBy`) & Partition Pruning**
3. **Task 2: Broadcast Join Optimization (`broadcast()`) vs Shuffle Joins**
4. **Task 3: Storage Format Size Benchmark (CSV vs Parquet vs Delta on 25,000+ Rows)**
5. **Task 4: Caching vs Persistence (`cache()`, `persist()`, StorageLevels)**
6. **Task 5: Delta Lake File Lifecycle & `VACUUM` Internals**
7. **Task 6: Delta Lake Liquid Clustering (`CLUSTER BY`) vs Traditional Partitioning**
8. **Task 7: Views Architecture: Local Temp View vs Global Temp View vs Semantic View**
9. **Task 8: Delta Live Tables (DLT) Declarative Medallion Pipelines & Data Quality Expectations**
10. **Task 9: Slowly Changing Dimensions (SCD Type 1 & SCD Type 2) in Pure PySpark (No SQL)**
11. **Task 10: Change Data Capture (CDC) & Delta Change Data Feed (CDF) Internals**
12. **Task 11: Optional Practice Environment Cleanup**

In [ ]:
# Task 0: Interactive Setup & Environment Configuration
# Configures widgets, storage paths (ADLS Gen2 with DBFS fallback), and imports necessary modules.

import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, current_date, expr, rand, when,
    broadcast, concat_ws, date_add, to_date, row_number, sha2
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, DateType, TimestampType, BooleanType
)
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel
from delta.tables import DeltaTable

# Widgets for interactive configuration
dbutils.widgets.text("storage_account", "azureprojectdevadls01", "ADLS Gen2 Storage Account")
dbutils.widgets.text("container", "raw", "Container Name")
dbutils.widgets.text("base_folder", "practice_optimization", "Base Folder in Container")
dbutils.widgets.text("target_catalog", "bronze_catalog", "Target Unity Catalog (or test)")
dbutils.widgets.text("target_schema", "practice", "Target Schema")

storage_account = dbutils.widgets.get("storage_account").strip()
container       = dbutils.widgets.get("container").strip()
base_folder     = dbutils.widgets.get("base_folder").strip()
target_catalog  = dbutils.widgets.get("target_catalog").strip()
target_schema   = dbutils.widgets.get("target_schema").strip()

# Primary ADLS Gen2 external path
adls_base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{base_folder}"
# Local DBFS fallback path (guaranteed to work in any cluster configuration)
dbfs_base_path = f"/tmp/{base_folder}"

# Test write access to determine if we use ADLS directly or fallback
try:
    dbutils.fs.ls(f"abfss://{container}@{storage_account}.dfs.core.windows.net/")
    active_base_path = adls_base_path
    print(f"[Connected] Using ADLS Gen2 Storage: {active_base_path}")
except Exception as e_storage:
    active_base_path = dbfs_base_path
    print(f"[Fallback Active] Using Workspace DBFS Storage: {active_base_path}")
    print(f"Reason: External storage credentials not mounted ({str(e_storage)[:80]}...)")

print(f"Target Catalog.Schema: {target_catalog}.{target_schema}")

In [ ]:
# Task 1: Partitioning - partitionBy & Partition Pruning
# Concept: Physical directory organization based on column values.
# Benefit: Query engines skip irrelevant partition folders (Partition Pruning).
# Interview Watchout: Avoid high-cardinality columns (e.g. Timestamp, UserID) -> Small Files Anti-Pattern!

partition_demo_path = f"{active_base_path}/partition_demo"

# 1. Create a synthetic sales dataset across multiple Years and Regions
df_sales = spark.range(1, 10001) \
    .withColumn("Year", when(col("id") % 3 == 0, 2024).when(col("id") % 3 == 1, 2025).otherwise(2026)) \
    .withColumn("Region", when(col("id") % 4 == 0, "North").when(col("id") % 4 == 1, "South").when(col("id") % 4 == 2, "East").otherwise("West")) \
    .withColumn("Amount", (rand() * 1000).cast("decimal(10,2)")) \
    .withColumn("TransactionDate", current_timestamp())

print("Sample Unpartitioned Data:")
df_sales.show(5, truncate=False)

# 2. Write to storage partitioned by Year and Region
print(f"Writing partitioned Delta table to: {partition_demo_path}...")
df_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("Year", "Region") \
    .save(partition_demo_path)

# 3. Inspect directory structure in storage
print("\n--- Physical Storage Directory Layout (Partition Folders) ---")
for year_folder in dbutils.fs.ls(partition_demo_path):
    if "Year=" in year_folder.name:
        print(f"  [Year Directory]: {year_folder.name}")
        for region_folder in dbutils.fs.ls(year_folder.path):
            if "Region=" in region_folder.name:
                print(f"      └── [Region Directory]: {region_folder.name}")

# 4. Demonstrate Partition Pruning with EXPLAIN plan
print("\n--- Query Plan Showing Partition Pruning (PartitionFilters) ---")
df_filtered = spark.read.format("delta").load(partition_demo_path) \
    .filter((col("Year") == 2025) & (col("Region") == "North"))

# Notice 'PartitionFilters' in the physical plan below:
df_filtered.explain(True)

In [ ]:
# Task 2: Broadcast Join Optimization
# Concept: In a standard join of two tables, Spark shuffles data across the network (SortMergeJoin).
# When joining a large Fact table with a small Dimension table (<10MB), broadcast() copies the
# small table to all executor nodes, eliminating the expensive shuffle stage entirely (BroadcastHashJoin).

# 1. Create a large Fact DataFrame (100,000 transactions)
df_fact = spark.range(1, 100001) \
    .withColumnRenamed("id", "TransactionID") \
    .withColumn("StoreID", (rand() * 10).cast("int") + 1) \
    .withColumn("SaleAmount", (rand() * 500).cast("decimal(10,2)"))

# 2. Create a small Dimension DataFrame (10 stores, ~few KB)
stores_data = [
    (1, "Downtown Store", "New York", 0.08875),
    (2, "Suburbs Plaza", "Chicago", 0.1025),
    (3, "Metro Hub", "Dallas", 0.0825),
    (4, "Tech Park", "San Francisco", 0.08625),
    (5, "Harbor View", "Seattle", 0.101),
    (6, "Central Market", "Boston", 0.0625),
    (7, "Airport Galleria", "Atlanta", 0.089),
    (8, "Valley Mall", "Phoenix", 0.086),
    (9, "Lakeside Center", "Minneapolis", 0.080),
    (10, "Mile High Outlet", "Denver", 0.0881)
]
df_stores = spark.createDataFrame(stores_data, ["StoreID", "StoreName", "City", "TaxRate"])

# 3. Standard Join Plan (Check if auto-broadcast triggers or force non-broadcast by disabling threshold)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)  # Temporarily disable auto-broadcast
df_standard_join = df_fact.join(df_stores, on="StoreID", how="inner")

print("=== Plan 1: Standard Join WITHOUT Broadcast (Notice Shuffle/SortMergeJoin) ===")
df_standard_join.explain()

# 4. Explicit Broadcast Join using broadcast()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)  # Reset default 10MB
df_broadcast_join = df_fact.join(broadcast(df_stores), on="StoreID", how="inner") \
    .withColumn("TotalWithTax", (col("SaleAmount") * (lit(1) + col("TaxRate"))).cast("decimal(10,2)"))

print("\n=== Plan 2: Optimized Broadcast Hash Join (Notice BroadcastHashJoin & No Shuffle!) ===")
df_broadcast_join.explain()

print("\nBroadcast Join Sample Output:")
df_broadcast_join.select("TransactionID", "StoreName", "City", "SaleAmount", "TotalWithTax").show(5)

In [ ]:
# Task 3: Storage Benchmark - CSV vs Parquet vs Delta Size Comparison on 25,000+ Rows
# Concept: Parquet/Delta use Columnar Storage, Dictionary Encoding, and Snappy Compression.
# CSV is row-based uncompressed text. We generate 25,000 realistic records and compare file sizes in ADLS.

benchmark_base = f"{active_base_path}/benchmark_size"
csv_path       = f"{benchmark_base}/data_csv"
parquet_path   = f"{benchmark_base}/data_parquet"
delta_path     = f"{benchmark_base}/data_delta"

# 1. Generate 25,000 rows of multi-column enterprise data (Strings, Numbers, Dates, Booleans, Long Text)
num_rows = 25000
print(f"Generating synthetic dataset with {num_rows:,} records...")

df_benchmark = spark.range(1, num_rows + 1) \
    .withColumnRenamed("id", "PolicyID") \
    .withColumn("CustomerName", concat_ws(" ", lit("Customer_Full_Name_Candidate_"), (col("PolicyID") % 500).cast("string"))) \
    .withColumn("PolicyType", when(col("PolicyID") % 4 == 0, "Comprehensive Automobile Protection")
                             .when(col("PolicyID") % 4 == 1, "Homeowners Standard Structural & Content")
                             .when(col("PolicyID") % 4 == 2, "Commercial Enterprise Liability Guarantee")
                             .otherwise("Individual Term Life & Disability Plan")) \
    .withColumn("State", when(col("PolicyID") % 5 == 0, "California")
                         .when(col("PolicyID") % 5 == 1, "Texas")
                         .when(col("PolicyID") % 5 == 2, "Florida")
                         .when(col("PolicyID") % 5 == 3, "New York")
                         .otherwise("Illinois")) \
    .withColumn("PremiumAmount", (rand() * 2500 + 300).cast("decimal(10,2)")) \
    .withColumn("CoverageLimit", (rand() * 500000 + 50000).cast("decimal(12,2)")) \
    .withColumn("IsActive", when(col("PolicyID") % 7 == 0, False).otherwise(True)) \
    .withColumn("EffectiveDate", date_add(to_date(lit("2024-01-01")), (col("PolicyID") % 700).cast("int"))) \
    .withColumn("UnderwriterNotes", concat_ws(" - ", lit("STANDARD_AUDIT_LOG_NOTE_INSPECTION_VERIFIED"), sha2(col("PolicyID").cast("string"), 256)))

# 2. Write in CSV, Parquet, and Delta formats
print("Writing dataset to CSV format...")
df_benchmark.write.mode("overwrite").option("header", "true").csv(csv_path)

print("Writing dataset to Parquet format (Snappy compressed)...")
df_benchmark.write.mode("overwrite").parquet(parquet_path)

print("Writing dataset to Delta Lake format...")
df_benchmark.write.mode("overwrite").format("delta").save(delta_path)

# 3. Helper function to compute total directory size in Bytes
def get_dir_size_bytes(dir_path):
    total_bytes = 0
    try:
        for f in dbutils.fs.ls(dir_path):
            if f.isDir():
                total_bytes += get_dir_size_bytes(f.path)
            else:
                total_bytes += f.size
    except Exception:
        pass
    return total_bytes

csv_bytes     = get_dir_size_bytes(csv_path)
parquet_bytes = get_dir_size_bytes(parquet_path)
delta_bytes   = get_dir_size_bytes(delta_path)

# 4. Print Size Comparison Report
print("\n" + "="*75)
print(f"  STORAGE FORMAT SIZE COMPARISON ({num_rows:,} ROWS)")
print("="*75)
print(f"{ 'Format':<15} | { 'Total Size (Bytes)':<20} | { 'Size (MB)':<12} | { 'Compression vs CSV':<20}")
print("-"*75)
print(f"{ 'CSV (Plain Text)':<15} | { csv_bytes:<20,} | { (csv_bytes/(1024*1024)):<12.2f} | { '0.00% (Baseline)':<20}")
print(f"{ 'Parquet (Snappy)':<15} | { parquet_bytes:<20,} | { (parquet_bytes/(1024*1024)):<12.2f} | { f'{(1 - parquet_bytes/csv_bytes)*100:.2f}% Smaller':<20}")
print(f"{ 'Delta Lake':<15} | { delta_bytes:<20,} | { (delta_bytes/(1024*1024)):<12.2f} | { f'{(1 - delta_bytes/csv_bytes)*100:.2f}% Smaller':<20}")
print("="*75)
print("Key Interview Takeaways:")
print("1. Parquet achieves massive compression via columnar storage, dictionary encoding & Snappy compression.")
print("2. Delta files share the same compressed Parquet data files plus a _delta_log transaction log directory.")
print("3. Querying Parquet reads ONLY requested columns (Projection Pruning), while CSV reads the entire file line by line!")

In [ ]:
# Task 4: Caching and Persistence (cache() vs persist())
# Concept:
# - cache(): Default shortcut for persist(StorageLevel.MEMORY_AND_DISK_DESER).
# - persist(StorageLevel): Explicit control over where data lives (Memory, Disk, Off-Heap, Serialized, Replicated).
# - Crucial Rule: Always unpersist() cached DataFrames when finished to avoid Executor Out-Of-Memory (OOM)!

# 1. Create a computationally heavy DataFrame (nested SHA-256 cryptographic hashes & math)
df_heavy = spark.range(1, 1000000) \
    .withColumn("Hash1", sha2(col("id").cast("string"), 256)) \
    .withColumn("Hash2", sha2(col("Hash1"), 256)) \
    .withColumn("CalculatedScore", expr("sqrt(id) * 3.14159"))

# Benchmark Action 1: WITHOUT Caching (recomputes the whole DAG each time)
t0 = time.time()
count1 = df_heavy.filter(col("CalculatedScore") > 500).count()
time_without_cache_1 = time.time() - t0

t0 = time.time()
count2 = df_heavy.filter(col("CalculatedScore") > 1000).count()
time_without_cache_2 = time.time() - t0

print(f"Run 1 (No Cache) : {time_without_cache_1:.3f} seconds")
print(f"Run 2 (No Cache) : {time_without_cache_2:.3f} seconds (Re-evaluated all transformations from scratch!)")

# 2. Apply persist() with specific StorageLevel
# Available levels: MEMORY_ONLY, MEMORY_AND_DISK, DISK_ONLY, MEMORY_AND_DISK_SER, etc.
df_heavy.persist(StorageLevel.MEMORY_AND_DISK)

# Materialize the cache with an initial action
t0 = time.time()
cached_count_init = df_heavy.count()
time_cache_materialize = time.time() - t0
print(f"\nCache Materialization Run : {time_cache_materialize:.3f} seconds (Cached to RAM/Disk)")

# Benchmark Action 2: Subsequent queries reading from CACHE
t0 = time.time()
cached_count_1 = df_heavy.filter(col("CalculatedScore") > 500).count()
time_with_cache_1 = time.time() - t0

t0 = time.time()
cached_count_2 = df_heavy.filter(col("CalculatedScore") > 1000).count()
time_with_cache_2 = time.time() - t0

print(f"Run 1 (With Cache) : {time_with_cache_1:.3f} seconds (Speedup: {time_without_cache_1 / max(time_with_cache_1, 0.001):.1f}x faster!)")
print(f"Run 2 (With Cache) : {time_with_cache_2:.3f} seconds")
print(f"Is DataFrame Currently Cached? -> {df_heavy.is_cached}")

# 3. Best Practice: Always Unpersist when done
df_heavy.unpersist()
print(f"After unpersist(): Is DataFrame Cached? -> {df_heavy.is_cached}")

In [ ]:
# Task 5: How VACUUM Works in Delta Lake
# Concept:
# Delta Lake uses Multi-Version Concurrency Control (MVCC). Updates/Deletes never delete files in-place;
# they write NEW Parquet files and flag older files as tombstoned in the _delta_log.
# VACUUM physically deletes files that are no longer referenced and are older than a retention threshold.
# Default safety retention: 168 hours (7 days).

vacuum_demo_path = f"{active_base_path}/vacuum_demo"

# 1. Create a fresh Delta Table (Version 0)
spark.range(1, 5) \
    .withColumn("Status", lit("ACTIVE")) \
    .write.format("delta").mode("overwrite").save(vacuum_demo_path)

tbl_delta = DeltaTable.forPath(spark, vacuum_demo_path)

# 2. Perform an Update (Version 1) -> generates new parquet file, leaves old file orphan/tombstoned
tbl_delta.update(condition=col("id") == 1, set={"Status": lit("SUSPENDED")})

# 3. Perform a Delete (Version 2) -> generates another file commit
tbl_delta.delete(condition=col("id") == 4)

# 4. Inspect physical files in the directory BEFORE Vacuum
print("--- Files on Storage BEFORE VACUUM (Notice multiple active & dead Parquet files) ---")
files_before = [f.name for f in dbutils.fs.ls(vacuum_demo_path) if f.name.endswith(".parquet")]
for f in files_before:
    print(f"  [Parquet File]: {f}")
print(f"Total Parquet file count before VACUUM: {len(files_before)}")

# 5. Disable the safety check temporarily to practice immediate retention (INTERVIEW ONLY / TEST ONLY!)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# 6. Run VACUUM DRY RUN (Audits files that will be permanently removed without deleting them)
print("\n--- Running VACUUM with DRY RUN (Inspection Mode) ---")
dry_run_df = tbl_delta.vacuum(0.0)  # 0 hours dry run
dry_run_df.show(truncate=False)

# 7. Run actual VACUUM to permanently purge dead files
print("Running actual VACUUM to delete tombstoned files...")
spark.sql(f"VACUUM delta.`{vacuum_demo_path}` RETAIN 0 HOURS")

# 8. Inspect physical files AFTER Vacuum
print("\n--- Files on Storage AFTER VACUUM (Dead files purged!) ---")
files_after = [f.name for f in dbutils.fs.ls(vacuum_demo_path) if f.name.endswith(".parquet")]
for f in files_after:
    print(f"  [Active Parquet File]: {f}")
print(f"Total Parquet file count after VACUUM: {len(files_after)}")

# Reset safety check
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

print("\nCritical Interview Note: Once VACUUM deletes files, Time Travel queries before that version will FAIL")
print("with 'FileNotFoundException' because the underlying Parquet files have been permanently erased!")

In [ ]:
# Task 6: How CLUSTER BY Works (Delta Liquid Clustering)
# Concept:
# Liquid Clustering is Databricks' modern replacement for both Hive-style Partitioning AND Z-Ordering.
# Key Advantages:
# 1. Avoids partition skew and small files.
# 2. Multi-dimensional clustering keys without folder hierarchy overhead.
# 3. Non-destructive: clustering keys can be altered anytime without table recreation (unlike PARTITION BY)!

cluster_demo_table = "practice_liquid_cluster_demo"
cluster_demo_path  = f"{active_base_path}/liquid_clustering_demo"

# 1. Create a table using CLUSTER BY (Liquid Clustering)
spark.sql(f"""
CREATE OR REPLACE TABLE delta.`{cluster_demo_path}` (
    CustomerID INT,
    CustomerName STRING,
    Region STRING,
    PolicyDate DATE,
    Premium DOUBLE
)
USING DELTA
CLUSTER BY (Region, CustomerID)
""")

print(f"Liquid Clustered table created at: {cluster_demo_path}")

# 2. Insert sample data into the clustered table
df_cluster_data = spark.range(1, 20000) \
    .withColumnRenamed("id", "CustomerID") \
    .withColumn("CustomerName", concat_ws(" ", lit("Customer_"), col("CustomerID").cast("string"))) \
    .withColumn("Region", when(col("CustomerID") % 3 == 0, "West").when(col("CustomerID") % 3 == 1, "East").otherwise("Central")) \
    .withColumn("PolicyDate", date_add(to_date(lit("2026-01-01")), (col("CustomerID") % 180).cast("int"))) \
    .withColumn("Premium", (rand() * 5000 + 500).cast("double"))

df_cluster_data.write.format("delta").mode("append").save(cluster_demo_path)

# 3. Run OPTIMIZE to trigger Liquid Clustering data re-organization
print("Running OPTIMIZE to compact and cluster data layout...")
optimize_res = spark.sql(f"OPTIMIZE delta.`{cluster_demo_path}`")
optimize_res.select("metrics.numFilesAdded", "metrics.numFilesRemoved", "metrics.dataSizeAdded").show(truncate=False)

# 4. Inspect clustering metadata with DESCRIBE DETAIL
print("--- Inspecting Table Detail for Clustering Properties ---")
detail_df = spark.sql(f"DESCRIBE DETAIL delta.`{cluster_demo_path}`")
detail_df.select("format", "clusteringColumns", "numFiles", "sizeInBytes").show(truncate=False)

print("Interview Tip: Liquid Clustering completely eliminates the need to run OPTIMIZE ... ZORDER BY.")
print("To change clustering keys on the fly, simply execute: ALTER TABLE ... CLUSTER BY (NewColumn1, NewColumn2)!")

In [ ]:
# Task 7: Views Architecture - Local Temp View vs Global Temp View vs Semantic View
# Comparison of all three view types in Apache Spark & Databricks Lakehouse.

# Create base dataset
df_customers = spark.createDataFrame([
    (1, "Alice", "Corporate", 125000.0, "Active"),
    (2, "Bob", "Retail", 45000.0, "Active"),
    (3, "Charlie", "Corporate", 89000.0, "Suspended"),
    (4, "Diana", "Retail", 62000.0, "Active")
], ["CustomerID", "CustomerName", "Segment", "AnnualRevenue", "Status"])

# 1. LOCAL TEMPORARY VIEW
# - Scope: Current SparkSession / Current Notebook only.
# - Lifecycle: Destroyed as soon as the notebook finishes or cluster session restarts.
df_customers.createOrReplaceTempView("vw_local_customers")
print("1. Local Temp View created: 'vw_local_customers'")
spark.sql("SELECT Segment, count(*) as Count, sum(AnnualRevenue) as TotalRev FROM vw_local_customers GROUP BY Segment").show()

# 2. GLOBAL TEMPORARY VIEW
# - Scope: Tied to a system-preserved database called 'global_temp'. Shared across multiple notebooks on the SAME cluster.
# - Lifecycle: Destroyed when the Spark cluster restarts.
df_customers.createOrReplaceGlobalTempView("vw_global_customers")
print("2. Global Temp View created in 'global_temp.vw_global_customers'")
spark.sql("SELECT * FROM global_temp.vw_global_customers WHERE Status = 'Active'").show()

# 3. SEMANTIC VIEW (Gold Layer / Unity Catalog / Metastore)
# - Scope: Globally accessible across the entire enterprise lakehouse (BI tools, PowerBI, Tableau, Analysts).
# - Lifecycle: Permanent metadata catalog asset (stored in Unity Catalog). Does not duplicate data storage!
# - Purpose: Enforces business logic (KPI calculations, column masking, row-level filters).

print("3. Demonstrating Semantic View Definition:")
semantic_view_sql = f"""
-- In production, created in Gold layer:
-- CREATE OR REPLACE VIEW {target_catalog}.{target_schema}.vw_customer_segment_kpi AS
SELECT 
    Segment,
    COUNT(CustomerID) AS TotalCustomers,
    SUM(AnnualRevenue) AS TotalRevenue,
    ROUND(AVG(AnnualRevenue), 2) AS AverageRevenuePerCustomer,
    ROUND(SUM(CASE WHEN Status = 'Active' THEN AnnualRevenue ELSE 0 END) / SUM(AnnualRevenue) * 100, 2) AS ActiveRevenueSharePct
FROM vw_local_customers
GROUP BY Segment
"""
df_semantic = spark.sql(semantic_view_sql)
df_semantic.show()

print("\n" + "="*80)
print(f"{'Feature':<22} | {'Local Temp View':<18} | {'Global Temp View':<18} | {'Semantic Gold View':<20}")
print("-"*80)
print(f"{'Scope':<22} | {'Current Session':<18} | {'Cluster-wide':<18} | {'Enterprise-wide (UC)':<20}")
print(f"{'Catalog Storage':<22} | {'In-Memory Only':<18} | {'global_temp DB':<18} | {'Metastore / Unity Catalog':<20}")
print(f"{'BI Tool Accessible?':<22} | {'No':<18} | {'No':<18} | {'Yes (PowerBI, Tableau)':<20}")
print(f"{'Primary Use Case':<22} | {'Intermediate wrangle':<18} | {'Notebook sharing':<18} | {'Single Source of Truth':<20}")
print("="*80)

In [ ]:
# Task 8: Delta Live Tables (DLT) Pipelines
# Concept:
# Delta Live Tables is a declarative ETL framework for building reliable data pipelines on Databricks.
# Key Features:
# 1. Declarative: You define 'what' tables to build, DLT builds the DAG and manages infrastructure.
# 2. Built-in Data Quality (Expectations): @dlt.expect, @dlt.expect_or_drop, @dlt.expect_or_fail.
# 3. Automatic Medallion Flow: Streaming and Batch unified.

# Below is the exact production-grade DLT pipeline code structure written in Python/PySpark:
dlt_pipeline_sample_code = '''
import dlt
from pyspark.sql.functions import col, current_timestamp, initcap, trim, lower

# --- 1. BRONZE LAYER: Raw Data Ingestion ---
@dlt.table(
    name="bronze_policies_raw",
    comment="Raw ingested policies from Landing ADLS Gen2 container",
    table_properties={"quality": "bronze"}
)
def bronze_policies_raw():
    return (
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "parquet")
            .load("abfss://landing@azureprojectdevadls01.dfs.core.windows.net/policies/")
    )

# --- 2. SILVER LAYER: Data Cleansing & Data Quality Expectations ---
@dlt.table(
    name="silver_policies_cleaned",
    comment="Cleansed policies with strict data quality constraints applied",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_fail("valid_policy_id", "PolicyID IS NOT NULL")
@dlt.expect_or_drop("valid_premium_amount", "Premium > 0")
@dlt.expect("warn_high_deductible", "Deductible <= 10000")  # Track metric in UI without dropping
def silver_policies_cleaned():
    return (
        dlt.read_stream("bronze_policies_raw")
            .withColumn("PolicyType", initcap(trim(col("PolicyType"))))
            .withColumn("CustomerEmail", lower(trim(col("CustomerEmail"))))
            .withColumn("SilverIngestedAt", current_timestamp())
    )

# --- 3. GOLD LAYER: Aggregated Business Semantic Reporting Table ---
@dlt.table(
    name="gold_policy_performance_kpi",
    comment="Aggregated policy metrics per State and Type for Executive BI dashboards",
    table_properties={"quality": "gold"}
)
def gold_policy_performance_kpi():
    return (
        dlt.read("silver_policies_cleaned")
            .groupBy("State", "PolicyType")
            .agg(
                count("PolicyID").alias("ActivePolicyCount"),
                sum("Premium").alias("TotalPremiumVolume"),
                avg("Premium").alias("AveragePremium")
            )
    )
'''

print("=== Delta Live Tables (DLT) Declarative Pipeline Architecture ===")
print(dlt_pipeline_sample_code)

print("\n--- DLT Expectation Action Modes for Interviews ---")
print("1. @dlt.expect('rule', 'expr')         -> Retains invalid record, logs metric failure to Event Log UI.")
print("2. @dlt.expect_or_drop('rule', 'expr') -> Drops dirty records automatically from downstream table.")
print("3. @dlt.expect_or_fail('rule', 'expr') -> Halts pipeline execution immediately on any critical violation!")

In [ ]:
# Task 9: Slowly Changing Dimensions (SCD Type 1 & SCD Type 2) in PURE PySpark (NO SQL!)
# Requirement: 100% PySpark DataFrame and DeltaTable API code without raw SQL strings.

scd1_path = f"{active_base_path}/scd_type1_demo"
scd2_path = f"{active_base_path}/scd_type2_demo"

# =========================================================================
# PART A: SCD TYPE 1 (In-Place Overwrite - No Historical Tracking)
# =========================================================================
print("\n" + "="*70)
print("  PART A: SCD TYPE 1 (OVERWRITE IN-PLACE) USING PYSPARK DELTA MERGE")
print("="*70)

# 1. Initial Dimension Table
dim_customers_init = spark.createDataFrame([
    (101, "Alice Brown", "Dallas", "TX", "2026-01-01"),
    (102, "Bob Miller", "Miami", "FL", "2026-01-01")
], ["CustomerID", "FullName", "City", "State", "LastModified"])

dim_customers_init.write.format("delta").mode("overwrite").save(scd1_path)
print("Initial SCD Type 1 Target Table:")
spark.read.format("delta").load(scd1_path).show()

# 2. Incoming CDC Batch (Alice moves to Austin; New customer Charlie added)
incoming_scd1 = spark.createDataFrame([
    (101, "Alice Brown", "Austin", "TX", "2026-02-15"), # City updated
    (103, "Charlie Davis", "Denver", "CO", "2026-02-15") # New Customer
], ["CustomerID", "FullName", "City", "State", "LastModified"])

# 3. Execute Pure PySpark MERGE
delta_scd1 = DeltaTable.forPath(spark, scd1_path)

delta_scd1.alias("tgt").merge(
    source=incoming_scd1.alias("src"),
    condition="tgt.CustomerID = src.CustomerID"
).whenMatchedUpdate(set={
    "City": col("src.City"),
    "State": col("src.State"),
    "LastModified": col("src.LastModified")
}).whenNotMatchedInsert(values={
    "CustomerID": col("src.CustomerID"),
    "FullName": col("src.FullName"),
    "City": col("src.City"),
    "State": col("src.State"),
    "LastModified": col("src.LastModified")
}).execute()

print("Target Table after SCD Type 1 MERGE (Notice Alice updated in-place to Austin):")
spark.read.format("delta").load(scd1_path).orderBy("CustomerID").show()

# =========================================================================
# PART B: SCD TYPE 2 (Full Historical Tracking with IsCurrent & Dates)
# =========================================================================
print("\n" + "="*70)
print("  PART B: SCD TYPE 2 (HISTORICAL TRACKING) USING PURE PYSPARK")
print("="*70)

# Schema: CustomerSK, CustomerID, FullName, City, EffectiveStartDate, EffectiveEndDate, IsCurrent
dim_scd2_init = spark.createDataFrame([
    (1, 101, "Alice Brown", "Dallas", to_date(lit("2025-01-01")), to_date(lit("9999-12-31")), True),
    (2, 102, "Bob Miller", "Miami", to_date(lit("2025-01-01")), to_date(lit("9999-12-31")), True)
], ["CustomerSK", "CustomerID", "FullName", "City", "EffectiveStartDate", "EffectiveEndDate", "IsCurrent"])

dim_scd2_init.write.format("delta").mode("overwrite").save(scd2_path)
print("Initial SCD Type 2 Table State (All records active):")
spark.read.format("delta").load(scd2_path).show(truncate=False)

# Incoming Change: Alice moves from Dallas to Austin on 2026-02-20; New Customer Charlie joins
incoming_scd2 = spark.createDataFrame([
    (101, "Alice Brown", "Austin"),
    (103, "Charlie Davis", "Denver")
], ["CustomerID", "FullName", "City"])

current_target_df = spark.read.format("delta").load(scd2_path)
today_date = to_date(lit("2026-02-20"))

# Step 1: Identify existing records where attributes have changed
active_target_df = current_target_df.filter(col("IsCurrent") == True)

df_joined = incoming_scd2.alias("inc").join(
    active_target_df.alias("tgt"),
    on="CustomerID",
    how="left"
)

# Records that require an update to existing row (close out old version)
rows_to_expire = df_joined.filter(
    col("tgt.CustomerID").isNotNull() & (col("inc.City") != col("tgt.City"))
).select("tgt.CustomerSK")

# Step 2: Expire historical records in target Delta table (Close out old version)
delta_scd2 = DeltaTable.forPath(spark, scd2_path)

delta_scd2.alias("tgt").merge(
    source=rows_to_expire.alias("exp"),
    condition="tgt.CustomerSK = exp.CustomerSK"
).whenMatchedUpdate(set={
    "EffectiveEndDate": today_date,
    "IsCurrent": lit(False)
}).execute()

# Step 3: Insert new active versions (both new customers + new versions of updated customers)
max_sk = current_target_df.agg({"CustomerSK": "max"}).collect()[0][0] or 0
w_sk = Window.orderBy("CustomerID")

rows_to_insert = df_joined.filter(
    col("tgt.CustomerID").isNull() | (col("inc.City") != col("tgt.City"))
).select(
    "inc.CustomerID",
    "inc.FullName",
    "inc.City"
).withColumn("CustomerSK", row_number().over(w_sk) + lit(max_sk)) \
 .withColumn("EffectiveStartDate", today_date) \
 .withColumn("EffectiveEndDate", to_date(lit("9999-12-31"))) \
 .withColumn("IsCurrent", lit(True))

# Append new active rows
rows_to_insert.select(
    "CustomerSK", "CustomerID", "FullName", "City", "EffectiveStartDate", "EffectiveEndDate", "IsCurrent"
).write.format("delta").mode("append").save(scd2_path)

print("Final SCD Type 2 Table State (Notice Alice has BOTH inactive history row and new active row!):")
spark.read.format("delta").load(scd2_path).orderBy("CustomerID", "CustomerSK").show(truncate=False)

In [ ]:
# Task 10: How CDC Works & Delta Lake Change Data Feed (CDF)
# Concept:
# Change Data Capture (CDC) captures row-level Inserts, Updates, and Deletes from source databases.
# Delta Lake's native CDC engine is called 'Change Data Feed' (CDF).
# When enabled, Delta automatically records pre-image and post-image values for all modifications!

cdc_demo_path = f"{active_base_path}/cdc_cdf_demo"

# 1. Create a Delta Table with Change Data Feed enabled (using Table Properties in PySpark)
df_cdc_init = spark.createDataFrame([
    (1, "Policy A", 1200.0),
    (2, "Policy B", 850.0)
], ["PolicyID", "PolicyName", "Premium"])

print("1. Creating Delta Table with 'delta.enableChangeDataFeed = true' (Version 0: INSERTS)...")
df_cdc_init.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .save(cdc_demo_path)

tbl_cdc = DeltaTable.forPath(spark, cdc_demo_path)

# 2. Perform an UPDATE (Version 1: Generates update_preimage and update_postimage)
print("2. Performing UPDATE on Policy 1 (Version 1)...")
tbl_cdc.update(
    condition=col("PolicyID") == 1,
    set={"Premium": col("Premium") + 350.0}
)

# 3. Perform a DELETE (Version 2: Generates delete)
print("3. Performing DELETE on Policy 2 (Version 2)...")
tbl_cdc.delete(condition=col("PolicyID") == 2)

# 4. Perform a new INSERT (Version 3: Generates insert)
print("4. Performing new INSERT for Policy 3 (Version 3)...")
spark.createDataFrame([(3, "Policy C", 2100.0)], ["PolicyID", "PolicyName", "Premium"]) \
    .write.format("delta").mode("append").save(cdc_demo_path)

# 5. Read the Change Data Feed across all versions using PySpark
print("\n" + "="*85)
print("  READING DELTA CHANGE DATA FEED (CDF) SYSTEM AUDIT LOG (Starting from Version 0)")
print("="*85)

df_cdf_changes = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .load(cdc_demo_path)

df_cdf_changes.select(
    "PolicyID", "PolicyName", "Premium",
    "_change_type", "_commit_version", "_commit_timestamp"
).show(truncate=False)

print("Key Interview Explanations for _change_type values:")
print("  • 'insert'            : Newly inserted records (Version 0 and Version 3).")
print("  • 'update_preimage'   : Row values BEFORE the update occurred (Old Premium = 1200.0).")
print("  • 'update_postimage'  : Row values AFTER the update occurred (New Premium = 1550.0).")
print("  • 'delete'            : Records that were deleted (Policy 2).")
print("\nDownstream consumers (e.g. Silver to Gold streaming pipelines) read ONLY these change feeds")
print("to process incremental updates without doing expensive full table scans!")

In [ ]:
# Task 11: Optional Practice Environment Cleanup
# Set cleanup_enabled = True when you wish to delete the temporary demo directories created in this notebook.

cleanup_enabled = False  # Toggle to True to purge practice folders

if cleanup_enabled:
    demo_paths = [
        partition_demo_path,
        benchmark_base,
        vacuum_demo_path,
        cluster_demo_path,
        scd1_path,
        scd2_path,
        cdc_demo_path
    ]
    print("Cleaning up temporary practice directories...")
    for p in demo_paths:
        try:
            dbutils.fs.rm(p, recurse=True)
            print(f"  [Deleted]: {p}")
        except Exception as e_del:
            print(f"  [Skip]: {p} ({e_del})")
    print("Cleanup completed successfully.")
else:
    print("Cleanup is disabled (cleanup_enabled = False).")
    print(f"All demo tables and storage paths remain preserved at: {active_base_path}")